# Week 12 Walkthrough — From Dictionaries to Classes

**Topic:** Classes, objects, __init__, self, inheritance, encapsulation

In Week 7 we counted circulation with dictionaries. It worked — but the rules about that data lived scattered across the program. We'll refactor it into classes, where data and the rules that govern it sit together. This is also the design you'll extend into a robot in Block 4.

Run the cells in order. Each step adds one idea to the program, and the last
section pulls the whole thing together. Change things and re-run — that is the
whole point of a notebook.

## Step 1 — Where dictionaries start to hurt


Nothing stops you writing a typo'd key, or a negative loan count, or forgetting
which fields a record is supposed to have.

In [ ]:
book = {"title": "Dune", "author": "Herbert", "loans": 42}

book["lons"] = 43            # typo - Python is perfectly happy
book["loans"] = -5           # nonsense - also fine as far as the dict knows

print(book)

## Step 2 — A class defines the shape once


`__init__` runs when you build one. `self` is the object being built.

In [ ]:
class Book:
    def __init__(self, title, author, loans=0):
        self.title = title
        self.author = author
        self.loans = loans


dune = Book("Dune", "Herbert", 42)
print(dune.title, dune.loans)

## Step 3 — __str__ makes it printable


Without it you get a memory address. Write one for every class.

In [ ]:
class Book:
    def __init__(self, title, author, loans=0):
        self.title = title
        self.author = author
        self.loans = loans

    def __str__(self):
        return f"{self.title} ({self.author}) - {self.loans} loans"


print(Book("Dune", "Herbert", 42))

## Step 4 — Methods put the rules next to the data


`check_out` can't be forgotten or done wrong, because it's the only way to do it.

In [ ]:
class Book:
    def __init__(self, title, author, loans=0):
        self.title = title
        self.author = author
        self.loans = loans
        self.on_shelf = True

    def check_out(self):
        if not self.on_shelf:
            return f"{self.title} is already out"
        self.on_shelf = False
        self.loans += 1
        return f"Checked out {self.title}"

    def check_in(self):
        self.on_shelf = True
        return f"Returned {self.title}"

    def __str__(self):
        where = "on shelf" if self.on_shelf else "on loan"
        return f"{self.title:16} {self.loans:3} loans  ({where})"


dune = Book("Dune", "Herbert", 42)
print(dune.check_out())
print(dune.check_out())
print(dune)

## Step 5 — Properties guard the invariants


"Loans can never be negative" is now enforced by the object itself, not by
whoever remembers.

In [ ]:
class Book:
    def __init__(self, title, loans=0):
        self.title = title
        self._loans = loans

    @property
    def loans(self):
        return self._loans

    @loans.setter
    def loans(self, value):
        if value < 0:
            raise ValueError("loans cannot be negative")
        self._loans = value


b = Book("Dune", 42)
b.loans = 43
print(b.loans)

try:
    b.loans = -5
except ValueError as e:
    print("refused:", e)

## Step 6 — Inheritance for the things that differ


A DVD is a library item that happens to have a shorter loan. Don't rewrite the
whole class — override the one method.

In [ ]:
class LibraryItem:
    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id

    def loan_period(self):
        return 21

    def __str__(self):
        return f"[{self.item_id}] {self.title} - {self.loan_period()} days"


class DVD(LibraryItem):
    def loan_period(self):
        return 7


class Reference(LibraryItem):
    def loan_period(self):
        return 0

    def __str__(self):
        return f"[{self.item_id}] {self.title} - library use only"


for item in [LibraryItem("Dune", "B1"), DVD("Arrival", "D1"), Reference("OED", "R1")]:
    print(item)

## Step 7 — Composition: a collection that owns its members


A branch **has** items. The branch class holds them and answers questions about
the set.

In [ ]:
class Branch:
    def __init__(self, name):
        self.name = name
        self.items = []

    def add(self, item):
        self.items.append(item)
        return self

    def total_loans(self):
        return sum(getattr(i, "loans", 0) for i in self.items)

    def __len__(self):
        return len(self.items)

    def __str__(self):
        return f"{self.name}: {len(self)} items"


main = Branch("Main")
main.add(Book("Dune", 42)).add(Book("Beloved", 93))
print(main, "-", main.total_loans(), "loans")

---

## The finished program

Everything above, in one place. This is the version worth keeping.


The Week 7 counter, rebuilt. Same output, but the rules now live inside the
objects — and adding a new item type doesn't touch the reporting code at all.

In [ ]:
# Week 12 - The circulation counter, as objects

class LibraryItem:
    """Anything that can be borrowed."""

    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id
        self._loans = 0
        self.on_shelf = True

    @property
    def loans(self):
        return self._loans

    def loan_period(self):
        return 21

    def check_out(self):
        if not self.on_shelf:
            return False
        self.on_shelf = False
        self._loans += 1
        return True

    def check_in(self):
        self.on_shelf = True

    def __str__(self):
        return f"{self.title:22} {self._loans:3} loans  ({self.loan_period()}d)"


class DVD(LibraryItem):
    def loan_period(self):
        return 7


class Reference(LibraryItem):
    def loan_period(self):
        return 0

    def check_out(self):
        return False                     # never leaves the building


class Branch:
    def __init__(self, name):
        self.name = name
        self.items = {}

    def add(self, item):
        self.items[item.title] = item

    def borrow(self, title):
        item = self.items.get(title)
        if item is None:
            return f"{title} is not held here"
        if item.check_out():
            item.check_in()              # returned immediately, for the demo
            return f"loaned {title}"
        return f"{title} could not be loaned"

    def report(self):
        ranked = sorted(self.items.values(), key=lambda i: i.loans, reverse=True)
        lines = [f"{self.name.upper()}", "=" * 40]
        lines += [str(i) for i in ranked]
        lines += ["-" * 40, f"total loans: {sum(i.loans for i in ranked)}"]
        return "\n".join(lines)


main = Branch("Main branch")
for item in (LibraryItem("Dune", "B1"), LibraryItem("Beloved", "B2"),
             DVD("Arrival", "D1"), Reference("Oxford English Dictionary", "R1")):
    main.add(item)

checkouts = ["Dune", "Beloved", "Dune", "Arrival", "Dune",
             "Oxford English Dictionary", "Beloved", "Missing Title"]

for title in checkouts:
    print(main.borrow(title))

print()
print(main.report())

---

## Try it yourself

Use the empty cells below. There is no grade attached — this is where the
learning actually happens.

**1.** Add a `Periodical` class with a 3-day loan period. Notice that `report()` needs no changes at all — that's polymorphism doing its job.

**2.** Give `LibraryItem` a `history` list that records each checkout, and print it for Dune.

**3.** Make `Branch.borrow` refuse when the item is already out, instead of returning it immediately.

**4.** Compare the finished cell to the Week 7 walkthrough. Which would you rather extend six months from now?

In [ ]:
# Try it yourself 1

In [ ]:
# Try it yourself 2

In [ ]:
# Try it yourself 3